<a href="https://colab.research.google.com/github/pierrot73/GenAIBootCamp/blob/Bootcamp/Week_6_Day2_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Daily Challenge

In [ ]:
%pip install --quiet datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [ ]:
pip install --upgrade datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 15.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pl

In [ ]:
from datasets import load_dataset
try:
    raw = load_dataset("ucirvine/sms_spam")
    print("Chargement réussi")
except Exception as e:
    print(f"Erreur : {e}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/359k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5574 [00:00<?, ? examples/s]

Chargement réussi


In [1]:
# 1. Installation des packages nécessaires (à exécuter dans le notebook ou en ligne de commande)
# %pip install --quiet datasets evaluate transformers[sentencepiece]

# 2. Chargement et inspection du dataset

from datasets import load_dataset  # Import de la fonction pour charger le dataset
import pandas as pd  # Pour manipuler les données sous forme de DataFrame

# Chargement du dataset SMS Spam via Hugging Face
raw = load_dataset("ucirvine/sms_spam")  # Dataset pré-chargé, divisé en 'train' et 'test'

# Sélectionner un sous-ensemble : 4000 pour train, 1000 pour validation
train_ds = raw['train'].select(range(4000))
val_ds = raw['train'].select(range(4000, 5000))

# Afficher les features pour voir les colonnes disponibles
print(train_ds.features)

# 3. Tokenization

from transformers import GPT2Tokenizer  # Import du tokenizer GPT-2

model_name = "gpt2"  # Nom du modèle GPT-2
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
# GPT-2 n'a pas de token pad par défaut, on le définit ici comme le token eos
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    # Tokenise les SMS avec padding et truncation
    return tokenizer(
        examples["sms"],  # La colonne contenant les messages
        padding="max_length",
        truncation=True,
        max_length=64
    )

# Appliquer la tokenisation sur l'ensemble d'entraînement et de validation
train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)

# 4. Initialisation du modèle

import torch
from transformers import GPT2ForSequenceClassification  # Import du modèle pour classification

# Charger GPT-2 avec une tête de classification binaire (spam ou ham)
model = GPT2ForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,  # 2 classes : spam et ham
    pad_token_id=tokenizer.eos_token_id
)

# 5. Définition des métriques

import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels)["precision"],
        "recall": recall.compute(predictions=preds, references=labels)["recall"],
        "f1": f1.compute(predictions=preds, references=labels)["f1"]
    }

# En contexte déséquilibré comme le dataset SMS spam, il est important de suivre:
# - La précision (precision) : combien de SMS prédits comme spam sont réellement spam (minimiser les faux positifs)
# - Le rappel (recall) : combien de spam sont correctement détectés (minimiser les faux négatifs)
# La précision seule peut être trompeuse si le modèle privilégie la majorité (ham). Le rappel indique la capacité à détecter tous les spam.
# L'accuracy peut être élevée même si le modèle ignore la classe minoritaire (spam), d'où l'intérêt de suivre aussi precision et recall.

# Si un modèle atteint une haute accuracy mais une faible recall pour la classe spam, cela signifie qu'il ne détecte pas bien tous les spam,
# ce qui peut laisser passer des messages indésirables. Il faut alors ajuster le seuil ou utiliser d'autres métriques pour améliorer la détection.

# 6. Configuration des paramètres d'entraînement

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_spam",
    do_train=True,
    do_eval=True,
    eval_steps=500,  # Evaluation toutes les 500 étapes
    save_steps=500,  # Sauvegarde toutes les 500 étapes
    logging_dir="./logs",
    logging_steps=500,  # Journalisation toutes les 500 étapes
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=5e-5,
    weight_decay=0.01,  # Décroissance du poids pour éviter le surapprentissage
)

# La weight_decay agit comme une régularisation pour réduire le surapprentissage.
# Une valeur plus élevée (ex: 0.1) peut renforcer la régularisation mais ralentir l'apprentissage.
# Une valeur plus faible (ex: 0.001) limite cet effet.

# 7. Entraînement et évaluation

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)

# Entraîner le modèle
trainer.train()

# Évaluer le modèle
metrics = trainer.evaluate()
print(metrics)
# On s'attend à voir des métriques comme eval_loss, eval_accuracy, etc.

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ValueError: Invalid pattern: '**' can only be an entire path component

Interprétation :

### Résultats d’évaluation :
```json
{
  'eval_loss': 0.0513,
  'eval_accuracy': 0.994,
  'eval_precision': 0.9854,
  'eval_recall': 0.9712,
  'eval_f1': 0.9783,
  'eval_runtime': 3.986 secondes,
  'eval_samples_per_second': 250.878,
  'eval_steps_per_second': 31.36,
  'epoch': 3.0
}
```

---

### Interprétation :

- **eval_loss (0.0513)** : La perte d’évaluation est très faible, ce qui indique que le modèle prédit très bien sur l’ensemble de validation/test.

- **eval_accuracy (99.4%)** : La précision est très élevée, presque parfaite, ce qui signifie que le modèle classe correctement la majorité des exemples.

- **eval_precision (98.5%)** : La précision indique que parmi les exemples que le modèle a prédits positifs, 98.5% sont corrects. Très bon score, peu de faux positifs.

- **eval_recall (97.1%)** : La sensibilité ou rappel indique que le modèle détecte 97.1% des vrais positifs. Très bon, peu de faux négatifs.

- **eval_f1 (97.8%)** : La moyenne harmonique entre précision et rappel est très haute, suggérant un modèle équilibré et performant.

- **eval_runtime (3.986 s)** : Le temps pour évaluer est inférieur à 4 secondes, ce qui est rapide.

- **eval_samples_per_second (250.878)** : Le modèle peut traiter environ 251 échantillons par seconde lors de l’évaluation.

- **eval_steps_per_second (31.36)** : Le modèle effectue environ 31 étapes d’évaluation par seconde.

- **epoch (3.0)** : Vous êtes à la fin de la troisième epoch.

---

### En résumé :
Le modèle fonctionne très bien, avec des performances presque parfaites sur l’ensemble d’évaluation. La faible perte, la précision, la recall et le F1 très élevés indiquent que le modèle est précis et fiable pour cette tâche.

---

### Sur la partie graphique :
Les données de la table montrent que la perte diminue au fil des étapes (500, 1000, 1500) :
- À 500 étapes : perte de 0.1665
- À 1000 étapes : perte de 0.0574
- À 1500 étapes : perte de 0.0302

Cela montre une convergence claire, la perte diminue avec l’entraînement, signe que le modèle apprend et s’améliore.

---
